In [1]:
!pip install transformers datasets torch accelerate -q

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!find /content/drive -name "*.csv"

/content/drive/MyDrive/DEEPGUARD/Dataset/WELFake_Dataset.csv


In [5]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/DEEPGUARD/Dataset/WELFake_Dataset.csv"
)

In [6]:
print(df.shape)

df.head()

(72134, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [7]:
df["label"].value_counts()

,count
label,
1,37106
0,35028


In [8]:
df = df.dropna()

print(df.shape)

(71537, 4)


In [9]:
df["content"] = df["title"] + " " + df["text"]

df = df[["content", "label"]]

df.head()

,content,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1
5,About Time! Christian Group Sues Amazon and SP...,1


In [10]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["content"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train:", len(train_texts))
print("Test:", len(test_texts))

Train: 57229
Test: 14308


In [11]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [13]:
print(len(train_texts))

57229


In [14]:
print(len(train_encodings["input_ids"]))
print(len(test_encodings["input_ids"]))

57229
14308


In [15]:
!nvidia-smi


Wed Jul  1 05:57:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/DEEPGUARD/Dataset/WELFake_Dataset.csv"
)

print(df.shape)

(72134, 4)


In [4]:
df = df.dropna()

df["content"] = df["title"] + " " + df["text"]

df = df[["content", "label"]]

print(df.shape)

(71537, 2)


In [5]:
from sklearn.model_selection import train_test_split

df_small = df.groupby("label").sample(
    n=5000,
    random_state=42
)

print(df_small.shape)

df_small["label"].value_counts()

(10000, 2)


,count
label,
0,5000
1,5000


In [6]:
print(df_small.shape)

df_small["label"].value_counts()

(10000, 2)


,count
label,
0,5000
1,5000


In [7]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_small["content"],
    df_small["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_small["label"]
)

print("Train:", len(train_texts))
print("Test:", len(test_texts))

Train: 8000
Test: 2000


In [8]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [9]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [10]:
print(len(train_encodings["input_ids"]))
print(len(test_encodings["input_ids"]))

8000
2000


In [11]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels.tolist()
})

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})


In [12]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=100,

    report_to="none"
)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.073343,0.071606,0.983500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.1198757038116455, metrics={'train_runtime': 117.4139, 'train_samples_per_second': 68.135, 'train_steps_per_second': 8.517, 'total_flos': 264934797312000.0, 'train_loss': 0.1198757038116455, 'epoch': 1.0})

In [17]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

print(confusion_matrix(test_labels, y_pred))

print(classification_report(test_labels, y_pred))

[[985  15]
 [ 18 982]]
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1000
           1       0.98      0.98      0.98      1000

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000



In [18]:
model.save_pretrained("deepguard_distilbert")
tokenizer.save_pretrained("deepguard_distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('deepguard_distilbert/tokenizer_config.json',
 'deepguard_distilbert/tokenizer.json')

In [19]:
!zip -r deepguard_distilbert.zip deepguard_distilbert

  adding: deepguard_distilbert/ (stored 0%)
  adding: deepguard_distilbert/tokenizer_config.json (deflated 43%)
  adding: deepguard_distilbert/config.json (deflated 49%)
  adding: deepguard_distilbert/tokenizer.json (deflated 71%)
  adding: deepguard_distilbert/model.safetensors (deflated 8%)


In [20]:
!ls

deepguard_distilbert  deepguard_distilbert.zip	drive  results	sample_data
